# Chapitre 8 - Exercices 8.5 à 8.7

* **8.5** - Généraliser les fonctions de perte de la figure 10.4 à plus de deux classes.
* **8.6** - Données de densité osseuse : spline de lissage, bandes fréquentistes,
  bayésiennes et bootstrap.
* **8.7** - L'algorithme EM vu comme un algorithme MM (minoration-maximisation).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch08_inference as inf
set_style()
rng = np.random.RandomState(0)

## Exercice 8.5 : Fonctions de perte pour plus de deux classes

**Énoncé.** Proposer des généralisations à plus de deux classes de chacune des fonctions
de perte de la figure 10.4 du livre, et concevoir un graphique adapté pour les comparer.

### Rappel : la figure 10.4 (cas à deux classes)

Avec $y \in \{-1, +1\}$, une fonction de score $f(x) \in \mathbb R$ et la règle
$\hat G(x) = \mathrm{signe}(f(x))$, toutes les pertes s'écrivent en fonction de la
**marge** $yf$ :

| perte | formule | rôle |
|---|---|---|
| erreur de classement | $\mathbb 1\{yf < 0\}$ | critère visé |
| exponentielle | $e^{-yf}$ | AdaBoost (chap. 10) |
| déviance binomiale | $\log(1 + e^{-2yf})$ | régression logistique |
| erreur quadratique | $(y - f)^2 = (1 - yf)^2$ | régression sur l'indicatrice |
| « support vector » | $(1 - yf)_+$ | SVM (chap. 12) |

Dans la figure 10.4, chaque perte est normalisée pour valoir 1 en $yf = 0$ (la déviance est
divisée par $\log 2$).

### Codage à $K$ classes

On utilise un **vecteur de scores** $f = (f_1, \dots, f_K)$ soumis à la contrainte
$\sum_k f_k = 0$ (sans elle, ajouter une constante à tous les $f_k$ ne changerait pas le
classement) et la règle $\hat G = \arg\max_k f_k$. La classe observée $k$ est codée par le
vecteur symétrique (codage de Zhu et al., repris pour AdaBoost multiclasse)

$$y_j = 1 \text{ si } j = k, \qquad y_j = -\tfrac{1}{K-1} \text{ sinon},
\qquad\text{de sorte que } \textstyle\sum_j y_j = 0.$$

Pour $K = 2$, $y = (\pm 1, \mp 1)$ et $f = (f_1, -f_1)$ : on retrouve le cadre à deux
classes avec $f = f_1$.

### Les généralisations proposées

| perte | généralisation à $K$ classes (classe observée $k$) | valeur en $f = 0$ |
|---|---|---|
| erreur de classement | $\mathbb 1\{\arg\max_j f_j \ne k\}$ | - |
| exponentielle | $\exp\big(-\tfrac1K\,y^Tf\big)$ | 1 |
| déviance multinomiale | $-\log p_k(f)$, $p_k = e^{f_k}/\sum_j e^{f_j}$ (éq. (4.17), (10.22)) | $\log K$ |
| erreur quadratique | $\sum_j (y_j - f_j)^2$ | $K/(K-1)$ |
| « support vector » (Lee, Lin, Wahba) | $\sum_{j\ne k}\big(f_j + \tfrac{1}{K-1}\big)_+$ | 1 |
| « support vector » (Crammer, Singer) | $\big(1 - [f_k - \max_{j\ne k}f_j]\big)_+$ | 1 |

Pour comparer, on normalise comme dans la figure 10.4 pour que chaque perte vaille 1 en
$f = 0$ (déviance divisée par $\log K$, erreur quadratique multipliée par $(K-1)/K$).

### Vérification : retour au cas $K = 2$

Avec $K = 2$, $y = (1, -1)$ et $f = (f_1, -f_1)$ :

* exponentielle : $y^Tf/2 = (f_1 + f_1)/2 = f_1$, donc $e^{-f_1}$ ✓ ;
* déviance : $p_1 = e^{f_1}/(e^{f_1} + e^{-f_1}) = 1/(1 + e^{-2f_1})$, donc
  $-\log p_1 = \log(1 + e^{-2f_1})$ ✓ ;
* quadratique : $(1 - f_1)^2 + (-1 + f_1)^2 = 2(1 - f_1)^2$, normalisée : $(1 - f_1)^2$ ✓ ;
* Lee-Lin-Wahba : $(f_2 + 1)_+ = (1 - f_1)_+$ ✓ ;
* Crammer-Singer : $(1 - 2f_1)_+$ : c'est la perte SVM à un changement d'échelle de $f$ près.

In [ ]:
def coding(k, K):
    y = np.full(K, -1.0 / (K - 1)); y[k] = 1.0
    return y

def losses(f, k):
    """Pertes multiclasses normalisées (valent 1 en f = 0). f : (n, K), classe observée k."""
    K = f.shape[1]
    y = coding(k, K)
    others = np.delete(np.arange(K), k)
    logp = f[:, k] - np.log(np.exp(f).sum(1))
    margin = f[:, k] - f[:, others].max(1)
    return {
        "erreur de classement": (f.argmax(1) != k).astype(float),
        "exponentielle": np.exp(-f @ y / K),
        "déviance multinomiale": -logp / np.log(K),
        "erreur quadratique": ((y - f) ** 2).sum(1) * (K - 1) / K,
        "SVM (Lee-Lin-Wahba)": np.clip(f[:, others] + 1 / (K - 1), 0, None).sum(1),
        "SVM (Crammer-Singer)": np.clip(1 - margin, 0, None),
    }

t = np.linspace(-2, 2, 9)
f2 = np.column_stack([t, -t])
L2 = losses(f2, 0)
print("K = 2, vérification des formules à deux classes :")
print("  exponentielle :", np.allclose(L2["exponentielle"], np.exp(-t)))
print("  déviance      :", np.allclose(L2["déviance multinomiale"], np.log1p(np.exp(-2 * t)) / np.log(2)))
print("  quadratique   :", np.allclose(L2["erreur quadratique"], (1 - t) ** 2))
print("  SVM (LLW)     :", np.allclose(L2["SVM (Lee-Lin-Wahba)"], np.clip(1 - t, 0, None)))

### Premier graphique : une coupe à une dimension

Le long de la direction du codage, $f = t\,y$ (on augmente le score de la bonne classe et
l'on diminue uniformément les autres), $t$ joue le rôle de la marge $yf$ : on obtient
l'analogue direct de la figure 10.4. Un calcul simple donne $e^{-t/(K-1)}$ pour
l'exponentielle, $(1 - t)^2$ pour la quadratique et $(1 - t)_+$ pour Lee-Lin-Wahba.